# XAI-MeteoFormer — training on Kaggle

**Run this with `Save & Run All (Commit)`, not interactively.** A committed
run keeps going on Kaggle's servers after you close the laptop; an
interactive session dies with the browser tab.

Before the first run:

1. Settings → Accelerator → **GPU P100** (or T4 x2)
2. Settings → Internet → **On** (needs phone verification; required for
   `git clone` and `pip install`)
3. Add Data → your `xai-meteoformer-data` dataset (the `data/processed/`
   folder uploaded from the laptop)
4. To continue a batch that ran out of the 12-hour session: Add Data →
   this notebook's own previous output. Finished runs are then skipped
   automatically.

Everything lands in `/kaggle/working/outputs/`, which becomes this
notebook's output and the input of the XAI and figures notebooks.

In [ ]:
# ------------------------------- CONFIG ------------------------------- #
REPO_URL   = "https://github.com/shapokok/xai-meteoformer.git"  # <-- yours
CODE_INPUT = None      # or "/kaggle/input/xai-meteoformer-code" for a private repo

DATA_INPUT = "/kaggle/input/xai-meteoformer-data"   # holds *_X.npy + *_meta.json
DATASET    = "jena"    # or "beijing_aotizhongxin"

MODELS = ["XAI-MeteoFormer", "DLinear", "PatchTST", "iTransformer",
          "LSTM", "Transformer", "Informer", "Autoformer"]
# TimesNet / TFT / Crossformer are the slow ones — run those on the 3070.

SEEDS   = [0, 1, 2, 3, 4]
EPOCHS  = 40

# P100/T4 have 16 GB, so these are roomier than the 8 GB numbers in the README.
BATCH = {"TimesNet": 32, "Crossformer": 32, "TFT": 64, "FEDformer": 32}
DEFAULT_BATCH = 128

# Stop launching new runs after this many hours so the notebook always
# gets to save its output before Kaggle's 12-hour cutoff.
DEADLINE_HOURS = 10.5

In [ ]:
import os, subprocess, sys, time, shutil, pathlib
T_START = time.time()

WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "xai-meteoformer"
TSLIB = WORK / "Time-Series-Library"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if CODE_INPUT:
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.copytree(CODE_INPUT, REPO)
elif not REPO.exists():
    sh(f"git clone --depth 1 {REPO_URL} {REPO}")

if not TSLIB.exists():
    sh(f"git clone --depth 1 https://github.com/thuml/Time-Series-Library.git {TSLIB}")

sh("pip install -q reformer_pytorch einops")

os.environ["TSLIB_PATH"] = str(TSLIB)
os.chdir(REPO)
print("cwd:", os.getcwd())
print("data:", sorted(os.listdir(DATA_INPUT)))

In [ ]:
# Resume: seed results.csv and checkpoints from a previous run of this
# notebook, if it was added as an input dataset. train.py skips any
# (model, dataset, seed) already present.
OUT = REPO / "outputs"
OUT.mkdir(exist_ok=True)

prev = None
for p in pathlib.Path("/kaggle/input").glob("*/outputs"):
    if (p / "results.csv").exists():
        prev = p
        break

if prev:
    print("resuming from", prev)
    shutil.copy(prev / "results.csv", OUT / "results.csv")
    for sub in ("checkpoints", "predictions"):
        if (prev / sub).exists():
            shutil.copytree(prev / sub, OUT / sub, dirs_exist_ok=True)
    import pandas as pd
    print(len(pd.read_csv(OUT / "results.csv")), "runs already done")
else:
    print("fresh start")

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

for model in MODELS:
    for seed in SEEDS:
        elapsed = (time.time() - T_START) / 3600
        if elapsed > DEADLINE_HOURS:
            print(f"\n== {elapsed:.1f} h elapsed, stopping so the output gets "
                  f"saved. Re-run this notebook with its own output added as "
                  f"an input to continue. ==")
            break
        cmd = [
            sys.executable, "src/train.py",
            "--dataset", DATASET,
            "--processed_dir", DATA_INPUT,
            "--out_dir", str(OUT),
            "--results_csv", str(OUT / "results.csv"),
            "--model_name", model,
            "--seed", str(seed),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH.get(model, DEFAULT_BATCH)),
            "--num_workers", "2",
        ]
        print("\n" + "=" * 70)
        print(" ".join(cmd))
        # subprocess per run: a crash or an OOM kills one run, not the batch,
        # and GPU memory is fully released between models.
        r = subprocess.run(cmd)
        if r.returncode != 0:
            print(f"!! {model} seed {seed} exited with {r.returncode}")
    else:
        continue
    break

print(f"\ntotal {(time.time() - T_START)/3600:.2f} h")

In [ ]:
import pandas as pd
df = pd.read_csv(OUT / "results.csv")
print(f"{len(df)} runs, {df.model.nunique()} models\n")

cols = [c for c in ["MAE", "RMSE", "R2", "CLS_AUC", "CLS_AP"] if c in df]
summary = (df[df.dataset == DATASET]
           .groupby("model")[cols]
           .agg(["mean", "std"])
           .round(4)
           .sort_values(("MAE", "mean")))
display(summary)

# copy outputs to the notebook root so they are easy to find in the output tab
shutil.copytree(OUT, WORK / "outputs", dirs_exist_ok=True)
print("saved to /kaggle/working/outputs")